In [1]:
import re
import pandas as pd
import numpy as np
from scipy.interpolate import CubicSpline, interp1d

print("--- Step 1: Loading Raw Data ---")
df_raw = pd.read_csv('/kaggle/input/competitions/finclub-open-project-26/dataset.csv')
df_sub = pd.read_csv('/kaggle/input/competitions/finclub-open-project-26/sandbox_solution.csv')

meta_cols = ['datetime', 'underlying_price']
option_cols = [col for col in df_raw.columns if col not in meta_cols]

def get_strike(col_name):
    numbers = re.findall(r'\d+', col_name)
    return int(numbers[-1]) if numbers else 0

sorted_option_cols = sorted(option_cols, key=get_strike)
df_sorted = df_raw[meta_cols + sorted_option_cols].copy()
strikes = np.array([get_strike(col) for col in sorted_option_cols])

print("--- Step 2: Relative 'Moneyness Space' Natural Spline ---")
def fit_moneyness_spline(row):
    # Extract the moving camera (Underlying Index Price) for this exact timestamp
    underlying = float(row['underlying_price'])
    
    # Isolate just the options data for math operations
    iv_values = row[sorted_option_cols].values.astype(float)
    known_mask = ~np.isnan(iv_values)
    num_known = np.sum(known_mask)
    
    if num_known == 0 or num_known == len(iv_values):
        return iv_values
        
    # CRITICAL UPGRADE: Transform absolute strikes into relative Moneyness coordinates
    moneyness_known = strikes[known_mask] / underlying
    moneyness_target = strikes / underlying
    y_known = iv_values[known_mask]
        
    try:
        if num_known >= 4:
            # Interpolate the stationary smile using relative coordinates
            cs = CubicSpline(moneyness_known, y_known, bc_type='natural', extrapolate=True)
            predicted_curve = cs(moneyness_target)
        else:
            interp_func = interp1d(moneyness_known, y_known, kind='linear', fill_value="extrapolate")
            predicted_curve = interp_func(moneyness_target)
            
        # Protect True Market Data
        predicted_curve[known_mask] = y_known
        return predicted_curve
    except:
        return iv_values

# Apply row-by-row
df_filled = pd.DataFrame(df_sorted.apply(fit_moneyness_spline, axis=1).tolist(), columns=sorted_option_cols)
df_final = pd.concat([df_sorted[meta_cols], df_filled], axis=1)

print("--- Step 3: Formatting Safely ---")
df_long = df_final.melt(id_vars=['datetime'], value_vars=sorted_option_cols, var_name='option_contract', value_name='value')
df_long['id'] = df_long['datetime'].astype(str) + '||' + df_long['option_contract']

# Map values directly without altering true decimals
lookup = dict(zip(df_long['id'], df_long['value']))
df_sub['value'] = df_sub['id'].map(lookup)

df_sub.to_csv('submission.csv', index=False)
print("✅ Moneyness Space Transformation applied! Surface is now dynamically anchored.")

--- Step 1: Loading Raw Data ---
--- Step 2: Relative 'Moneyness Space' Natural Spline ---
--- Step 3: Formatting Safely ---
✅ Moneyness Space Transformation applied! Surface is now dynamically anchored.


In [2]:
import re
import pandas as pd
import numpy as np
from scipy.interpolate import CubicSpline, interp1d
from scipy.signal import savgol_filter

print("--- Step 1: Loading Raw Data ---")
df_raw = pd.read_csv('/kaggle/input/competitions/finclub-open-project-26/dataset.csv')
df_sub = pd.read_csv('/kaggle/input/competitions/finclub-open-project-26/sandbox_solution.csv')

meta_cols = ['datetime', 'underlying_price']
option_cols = [col for col in df_raw.columns if col not in meta_cols]

def get_strike(col_name):
    numbers = re.findall(r'\d+', col_name)
    return int(numbers[-1]) if numbers else 0

sorted_option_cols = sorted(option_cols, key=get_strike)
df_sorted = df_raw[meta_cols + sorted_option_cols].copy()
strikes = np.array([get_strike(col) for col in sorted_option_cols])

print("--- Step 2: Spatial Interpolation (Relative Moneyness Spline) ---")
def fit_moneyness_spline(row):
    underlying = float(row['underlying_price'])
    iv_values = row[sorted_option_cols].values.astype(float)
    known_mask = ~np.isnan(iv_values)
    num_known = np.sum(known_mask)
    
    if num_known == 0 or num_known == len(iv_values):
        return iv_values
        
    moneyness_known = strikes[known_mask] / underlying
    moneyness_target = strikes / underlying
    y_known = iv_values[known_mask]
        
    try:
        if num_known >= 4:
            cs = CubicSpline(moneyness_known, y_known, bc_type='natural', extrapolate=True)
            predicted_curve = cs(moneyness_target)
        else:
            interp_func = interp1d(moneyness_known, y_known, kind='linear', fill_value="extrapolate")
            predicted_curve = interp_func(moneyness_target)
            
        predicted_curve[known_mask] = y_known
        return predicted_curve
    except:
        return iv_values

# Apply row-by-row to construct the spatial base
df_filled = pd.DataFrame(df_sorted.apply(fit_moneyness_spline, axis=1).tolist(), columns=sorted_option_cols)

print("--- Step 3: Temporal Smoothing (Savitzky-Golay Filter) ---")
df_smoothed = df_filled.copy()

# Sweep vertically down the timeline for each specific option contract to eliminate bid-ask bounce
for col in sorted_option_cols:
    if len(df_smoothed) > 5:
        # window_length=5, polyorder=2 is the quant standard for removing micro-jitter without losing trend shape
        df_smoothed[col] = savgol_filter(df_smoothed[col], window_length=5, polyorder=2)

# CRITICAL: The filter alters all data, so we must re-inject the true market observations
raw_matrix = df_sorted[sorted_option_cols].values
smoothed_matrix = df_smoothed.values
known_mask = ~np.isnan(raw_matrix)

# Overlay the exact known values back on top to guarantee 0 penalty on true data
smoothed_matrix[known_mask] = raw_matrix[known_mask]
df_final_options = pd.DataFrame(smoothed_matrix, columns=sorted_option_cols)
df_final = pd.concat([df_sorted[meta_cols], df_final_options], axis=1)

print("--- Step 4: Formatting Safely ---")
df_long = df_final.melt(id_vars=['datetime'], value_vars=sorted_option_cols, var_name='option_contract', value_name='value')
df_long['id'] = df_long['datetime'].astype(str) + '||' + df_long['option_contract']

lookup = dict(zip(df_long['id'], df_long['value']))
df_sub['value'] = df_sub['id'].map(lookup)

df_sub.to_csv('submission.csv', index=False)
print("✅ Spatio-Temporal matrix complete! Time-axis jitter eliminated.")

--- Step 1: Loading Raw Data ---
--- Step 2: Spatial Interpolation (Relative Moneyness Spline) ---
--- Step 3: Temporal Smoothing (Savitzky-Golay Filter) ---
--- Step 4: Formatting Safely ---
✅ Spatio-Temporal matrix complete! Time-axis jitter eliminated.
